# TP · etudiants · Google Colab

Cybersup · Machine Learning Avancé · **Chrys Fé-Marty NIONGOLO** · septembre 2026

Ce notebook est autonome : aucun clonage, jeton GitHub ou fichier du dépôt n'est nécessaire.
Dans Colab : **Fichier → Importer le notebook**, puis exécuter la cellule d'installation
dans une nouvelle session avant les autres cellules. Python 3.12 ou ultérieur requis.
Si Colab demande de redémarrer la session après installation, accepter puis reprendre
depuis le début. Enregistrer une copie du notebook et télécharger ses sorties avant de quitter.

Le CPU suffit. Ces estimateurs scikit-learn ne deviennent pas des modèles GPU en sélectionnant un accélérateur.
Internet est nécessaire pour installer les bibliothèques; les données sont synthétiques.
Les ressources Colab ne sont pas garanties : [FAQ officielle](https://research.google.com/colaboratory/faq.html).
Les versions ci-dessous sont celles du cours, pas une recommandation de toujours installer les dernières.


In [ ]:
# Exécuter en premier, dans une session neuve.
import os, sys, subprocess
from importlib.metadata import version as package_version
if sys.version_info < (3, 12):
    raise RuntimeError("Choisir un runtime Python >= 3.12, ou utiliser le pack local Python 3.12.")
required = {'numpy': '2.5.3', 'pandas': '3.0.6', 'scipy': '1.18.1', 'scikit-learn': '1.9.1', 'matplotlib': '3.11.2'}
if os.environ.get("CYBERSUP_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *[f"{k}=={v}" for k, v in required.items()]])
for key, expected in required.items():
    assert package_version(key) == expected, f"Version incorrecte : {key}. Reprendre dans une session neuve."
for key in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[key] = "2"
print("Python", sys.version.split()[0], "· dépendances", {k: package_version(k) for k in required})


# TP 02 · Arbres, bagging et boosting
90 minutes. Régression synthétique Friedman : relations non linéaires connues, sans prétention métier.
Livrable : RMSE de validation, temps d'ajustement et courbe du boosting. Choisir un compromis argumenté.
Source : https://scikit-learn.org/stable/modules/ensemble.html

Pendant la séance : 70 min de TP + 20 min de démonstration 08 XGBoost. Le stacking est une extension après le cours si nécessaire.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_friedman1
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.metrics import root_mean_squared_error
X, y = make_friedman1(n_samples=1000, n_features=10, noise=1.5, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)
models = {"Ridge": make_pipeline(StandardScaler(), Ridge(alpha=10)),
          "Arbre": DecisionTreeRegressor(max_depth=5, random_state=42),
          "Forêt": RandomForestRegressor(n_estimators=100, min_samples_leaf=3, n_jobs=1, random_state=42),
          "Boosting": HistGradientBoostingRegressor(max_iter=100, max_leaf_nodes=15, random_state=42)}
cv = KFold(3, shuffle=True, random_state=42)
rows = []
for name, model in models.items():
    r = cross_validate(model, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error")
    rows.append([name, -r["test_score"].mean(), r["fit_time"].mean()])
print(pd.DataFrame(rows, columns=["modèle", "RMSE CV", "secondes fit"]))

Exercice 1 (25 min). Tracer RMSE train/validation pour les étapes successives d'un boosting. Où arrêter ? Pourquoi ce choix ne doit-il pas utiliser le test ?

In [ ]:
# Votre réponse / votre code ici.
# Les exemples guidés restent exécutables.

Exercice 2 (20 min). Réduire max_features de la forêt de 1 à 0.6. Comparer sur les mêmes plis. La décorrélation améliore-t-elle toujours le score ?

In [ ]:
# Votre réponse / votre code ici.
# Les exemples guidés restent exécutables.

Extension (20 min). Construire un stacking avec des prédictions hors pli pour le méta-modèle. Comparer son coût à son bénéfice.

In [ ]:
# Votre réponse / votre code ici.
# Les exemples guidés restent exécutables.